In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt', force=True)
nltk.download('stopwords', force=True)
nltk.download('wordnet', force=True)
nltk.download('vader_lexicon', force=True)
nltk.download('averaged_perceptron_tagger', force=True)
nltk.download('punkt_tab', force=True)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sbhalerao\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [4]:
pain_points = '../output/cleaned_challenges.csv'
expectations = '../output/cleaned_expectations.csv'

df_pain_points = pd.read_csv(pain_points)
df_expectations = pd.read_csv(expectations)

In [9]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]

    lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
    return ' '.join(lemmas)

df_pain_points['processed_pain_points'] = df_pain_points['Pain Point'].apply(preprocess_text)
print(df_pain_points[['Focus Group', 'Pain Point', 'processed_pain_points']].head(10))

                         Focus Group  \
0  Assessment Assessment Supervisors   
1  Assessment Assessment Supervisors   
2  Assessment Assessment Supervisors   
3  Assessment Assessment Supervisors   
4  Assessment Assessment Supervisors   
5  Assessment Assessment Supervisors   
6  Assessment Assessment Supervisors   
7  Assessment Assessment Supervisors   
8          CPO Central Permit Office   
9          CPO Central Permit Office   

                                          Pain Point  \
0  Historical Record Management: Historical permi...   
1  Incomplete Property History: IPS does not alwa...   
2  Large Plan Viewing: Large plans and surveys ca...   
3  Limited Historical Visibility: Historical reco...   
4  Property Research Complexity: Assessment work ...   
5  Subdivision Tracking: Following parcel history...   
6  Permit Dependency: Assessment activities depen...   
7  Seasonal Workload: Assessment activities vary ...   
8  Split Permit & Code Systems: Permit and Code E...   

In [12]:
df_expectations['processed_expectations'] = df_expectations['Expectation'].apply(preprocess_text)
print(df_expectations[['Focus Group', 'Expectation', 'processed_expectations']].head(10))

                         Focus Group  \
0  Assessment Assessment Supervisors   
1  Assessment Assessment Supervisors   
2  Assessment Assessment Supervisors   
3  Assessment Assessment Supervisors   
4  Assessment Assessment Supervisors   
5                BAA BAA Supervisors   
6                BAA BAA Supervisors   
7                BAA BAA Supervisors   
8                BAA BAA Supervisors   
9                BAA BAA Supervisors   

                                         Expectation  \
0  Improved Historical Records: Maintain comprehe...   
1  Integrated Property Information: Better integr...   
2  Improved Document Viewing: Improve the ability...   
3  Better Property Research: Make historical prop...   
4  Improved Parcel Tracking: Better track parcel ...   
5  Automated Ticket Processing: Automate ticket c...   
6  Linked Property Cases: Automatically link all ...   
7  Improved Evidence Management: Provide a centra...   
8  Enhanced Search Capabilities: Improve searchin...   

In [13]:
df_pain_points.shape, df_expectations.shape

((322, 3), (287, 3))

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter

# Rebuild processed_content here if the preprocessing cell has not been run.
if 'processed_content' not in df_pain_points.columns:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        tokens = word_tokenize(str(text))
        filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]
        lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        return ' '.join(lemmas)

    df_pain_points['processed_pain_points'] = df_pain_points['Pain Point'].apply(preprocess_text)

if 'processed_expectations' not in df_expectations.columns:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        tokens = word_tokenize(str(text))
        filtered_tokens = [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]
        lemmas = [lemmatizer.lemmatize(token) for token in filtered_tokens]
        return ' '.join(lemmas)

    df_expectations['processed_expectations'] = df_expectations['Expectation'].apply(preprocess_text)


def get_category(text):
    tokens = str(text).split()
    if not tokens:
        return 'unknown'
    return Counter(tokens).most_common(1)[0][0]

sia = SentimentIntensityAnalyzer()

def get_sentiment_label(text):
    text = str(text)
    compound_score = sia.polarity_scores(text)['compound']
    token_count = len([token for token in text.split() if token.strip()])

    if token_count == 1 and -0.05 < compound_score < 0.05:
        return 'neutral'
    if compound_score >= 0.05:
        return 'positive'
    if compound_score <= -0.05:
        return 'negative'
    return 'neutral'

# Build a label column from the cleaned text so the model has a target to learn.
df_pain_points['label'] = df_pain_points['processed_pain_points'].apply(get_sentiment_label)
df_pain_points = df_pain_points[df_pain_points['label'].isin(['positive', 'negative', 'neutral'])].copy()

# Create a category from the dominant word in the cleaned content.
df_pain_points['category'] = df_pain_points['processed_pain_points'].apply(get_category)
df_expectations['category'] = df_expectations['processed_expectations'].apply(get_category)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df_pain_points['processed_pain_points'])
y = df_pain_points['label'].map({'positive': 1, 'negative': 0, 'neutral': 2}).values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(df_pain_points[['Pain Point', 'processed_pain_points', 'category', 'label']].head(10))
print(df_expectations[['Focus Group', 'Expectation', 'processed_expectations', 'category']].head(10))

Feature matrix shape: (322, 601)
Target vector shape: (322,)
                                          Pain Point  \
0  Historical Record Management: Historical permi...   
1  Incomplete Property History: IPS does not alwa...   
2  Large Plan Viewing: Large plans and surveys ca...   
3  Limited Historical Visibility: Historical reco...   
4  Property Research Complexity: Assessment work ...   
5  Subdivision Tracking: Following parcel history...   
6  Permit Dependency: Assessment activities depen...   
7  Seasonal Workload: Assessment activities vary ...   
8  Split Permit & Code Systems: Permit and Code E...   
9  Scattered Information: Property and permit inf...   

                               processed_pain_points     category     label  
0  historical record management historical permit...   historical   neutral  
1  incomplete property history ip always provide ...     property   neutral  
2  large plan viewing large plan survey viewed ef...        large  positive  
3  limited